In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate, KFold
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import RFE
from sklearn import set_config
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_predict

In [ ]:
!pip install dagshub
!pip install mlflow

In [ ]:
import dagshub
dagshub.init(repo_owner='icosahedron31', repo_name='ML-HW-1', mlflow=True)

import mlflow


In [ ]:
df=pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
df

In [ ]:
df.info()

In [ ]:
from sklearn.model_selection import train_test_split

target_col = 'SalePrice'
X = df.drop(columns=[target_col])
y = df[target_col]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
cat_cols = [col for col in X_train.columns if X_train[col].dtype == 'object']
num_cols = [col for col in X_train.columns if X_train[col].dtype != 'object']

print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")
print(f"Numerical columns ({len(num_cols)}): {num_cols}")

In [ ]:
df.info()

In [ ]:
!pip install matplotlib

In [ ]:
import dagshub
import mlflow
dagshub.init(repo_owner='icosahedron31', repo_name='ML-HW-1', mlflow=True)
mlflow.set_tracking_uri("https://dagshub.com/icosahedron31/ML-HW-1.mlflow")

#with mlflow.start_run():
 # mlflow.log_param('parameter name', 'value')
 # mlflow.log_metric('metric name', 1)

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class NAPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, threshold):
       self.threshold = threshold
    def fit(self, X, y):
        # Generate Fill Na Values Just in Case
        self.columnsToDrop = []
        for col in X.columns : 
            if X[col].isna().mean() > self.threshold : 
                self.columnsToDrop.append(col)
        return self


    def transform(self, X):
        X_transformed = X.copy()
        X_transformed=X_transformed.drop(columns=self.columnsToDrop)
        return X_transformed

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class CustomPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, mean_columns, one_hot_columns):
        self.mean_columns = mean_columns # Columns Which Should Be Preprocessed Using WOE
        self.one_hot_columns = one_hot_columns # Columns Which Should Be Preprocessed Using One Hot Encoder
        self.one_hot_expected = {}
    def fit(self, X, y):
        #
        self.mean_columns = [col for col in self.mean_columns if col in X.columns]
        self.one_hot_columns = [col for col in self.one_hot_columns if col in X.columns]
        # Generate Fill Na Values Just in Case
        self.mean_columns_fill_na = X[self.mean_columns].mode().T[0].to_dict()
        df_mean = X.copy()
        target_col = 'SalePrice'
        df_mean[target_col] = y
        self.global_mean = y.mean()
        self.mean_mappings = {}
        self.one_hot_mappings = {}
        for col in self.mean_columns:
         
            groups = df_mean.groupby([col])[target_col].agg(['mean'])
            groups.columns = ['mean']
            groups.fillna(0, inplace=True)
            mean_dict = groups['mean'].to_dict()
            self.mean_mappings[col] = mean_dict
        for col in self.one_hot_columns : 
            unique_vals = X[col].dropna().unique()
            self.one_hot_expected[col] = [f"{col}_{val}" for val in unique_vals]
        return self


    def transform(self, X):
        X_transformed = X.copy()

        
        for col in self.mean_columns:
            X_transformed[f'{col}_mean'] = X_transformed[col].map(self.mean_mappings[col]).fillna(self.global_mean)
            X_transformed.drop(columns=col, inplace=True)

      
        for col in self.one_hot_columns:
            dummies = pd.get_dummies(
                X_transformed[col], 
                prefix=col, 
                dummy_na=True, 
                dtype=int
            )
            for expected_col in self.one_hot_expected[col]:
                if expected_col not in dummies.columns:
                    dummies[expected_col] = 0
            dummies = dummies[self.one_hot_expected[col]]
            X_transformed = pd.concat([
                X_transformed.drop(col, axis=1), 
                dummies
            ], axis=1)

        n = X_transformed.isna().mean()

        na_cols = list(n[n > 0].index)

  

        for col in na_cols:

            if not '_' in col : continue
            name, pr = col.split("_", 1)
            if pr != "mean":
                print("Error Related to Nans")

            dic = self.mean_columns_fill_na
            mappings = self.mean_mappings
           
            X_transformed[col] = X_transformed[col].fillna(mappings[name][dic[name]])
            #else X_transformed[col]
          
            
        return X_transformed

In [ ]:
from sklearn.model_selection import train_test_split

target_col = 'SalePrice'
X = df.drop(columns=[target_col])
y = df[target_col]
#
X_train, y_train = X, y

In [ ]:
s = X_train[cat_cols].nunique()
s

In [ ]:
threshold = 3
mean_columns = s[s > threshold].index
one_hot_columns = s[s<=threshold].index
mean_columns, one_hot_columns



In [ ]:
preprocessor = CustomPreprocessor(mean_columns, one_hot_columns)
X_train_transformed = preprocessor.fit_transform(X_train, y_train)
X_train_transformed.info()

In [ ]:
pipeline = Pipeline([
    
    ('categorical_transformer', CustomPreprocessor(mean_columns, one_hot_columns)),
    ('imputer', SimpleImputer(strategy='median')),  
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

In [ ]:
import numpy as np
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    median_absolute_error
)
import matplotlib.pyplot as plt
def compute_metrics(y, y_pred):
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    mae = mean_absolute_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    med_ae = median_absolute_error(y, y_pred)

    nrmse = rmse / np.std(y)

    return {
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "median_ae": med_ae,
        "nrmse_std": nrmse,
    }
def plot_actual_vs_predicted(y_val, y_pred):
    fig, ax = plt.subplots()
    ax.scatter(y_val, y_pred, alpha=0.2)
    ax.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--')
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    ax.set_title("Actual vs Predicted")
    return fig

def plot_residuals(y_val, y_pred):
    residuals = (y_val - y_pred) / y_val
    fig, ax = plt.subplots()
    ax.scatter(y_pred, residuals, alpha=0.2)
    ax.axhline(0, color='r', linestyle='--')
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Residuals")
    ax.set_title("Residuals Plot")
    return fig

def plot_residuals_distribution(y_val, y_pred):
    residuals = (y_val - y_pred) / y_val
    fig, ax = plt.subplots()
    ax.hist(residuals, bins=100, edgecolor='black')
    ax.axvline(0, color='r', linestyle='--')
    ax.set_xlabel("Residual")
    ax.set_title("Residuals Distribution")
    return fig

def plot_feature_importance(model, feature_names):
    fig, ax = plt.subplots()
    importance = pd.Series(model.feature_importances_, index=feature_names)
    importance.sort_values().plot(kind='barh', ax=ax)
    ax.set_title("Feature Importance")
    return fig

In [ ]:
from sklearn.base import clone
def start_mlflow_run(name, grid_search_arg, X, y, run_name=None):
    mlflow.set_experiment(name)
    
    with mlflow.start_run(run_name=run_name):  
        y_log = np.log1p(y)
        grid_search_arg.fit(X, y_log)
        best_model = grid_search_arg.best_estimator_
        mlflow.log_params(grid_search_arg.best_params_)
        kf = grid_search_arg.cv

        preds = np.zeros_like(y, dtype=float)
        i = 0
        for train_idx, val_idx in kf.split(X):
            model = clone(best_model)
            
            model.fit(X.iloc[train_idx], y_log.iloc[train_idx])
            
            pred_log = model.predict(X.iloc[val_idx])
            preds[val_idx] = np.expm1(pred_log)
            mlflow.log_metric(f"split {i} RMSE", np.sqrt(np.mean((preds[val_idx] - y.iloc[val_idx])**2)))
            i += 1
        # metrics
        n_splits = grid_search_arg.cv.get_n_splits()
        train_scores = [
            -grid_search_arg.cv_results_[f"split{i}_train_score"][grid_search_arg.best_index_]
            for i in range(n_splits)
        ]
        
        val_scores = [
            -grid_search_arg.cv_results_[f"split{i}_test_score"][grid_search_arg.best_index_]
            for i in range(n_splits)
        ]
        mlflow.log_metric("cv_mse_train_mean", np.mean(train_scores))
        mlflow.log_metric("cv_mse_val_mean", np.mean(val_scores))
        mlflow.log_metric("cv_mse_std", np.std(val_scores))


        # plots in real scale
        y_pred_log = cross_val_predict(
            grid_search_arg.best_estimator_,
            X,
            y_log,
            cv=grid_search_arg.cv,
            n_jobs=-1
        )
        y_pred = np.expm1(y_pred_log)
        y_real = y
        fig1 = plot_actual_vs_predicted(y_real, y_pred)
        fig2 = plot_residuals(y_real, y_pred)
        fig3 = plot_residuals_distribution(y_real, y_pred)
        mlflow.log_figure(fig1, "plots/actual_vs_predicted.png")
        mlflow.log_figure(fig2, "plots/residuals.png")
        mlflow.log_figure(fig3, "plots/residuals_distribution.png")
        plt.close('all')

        mlflow.sklearn.log_model(best_model, "model")

In [ ]:
param_grid = {

    'imputer__strategy': ['mean', 'median', 'most_frequent'],
}

grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='neg_mean_squared_error',
    return_train_score=True,
    n_jobs=-1,
    verbose=2
)

In [ ]:

import mlflow
import mlflow.sklearn
import numpy as np
start_mlflow_run("KFold_simple_model", grid_search,X_train, y_train, "Kfold_cross_val_without_feature_selection")

Correlation filter

In [ ]:
X_corr = X_train_transformed.copy()
X_corr['target'] = y_train

In [ ]:

corr_matrix = X_corr.corr().abs()

In [ ]:
corr_matrix

In [ ]:
# Create a mask for the upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Find feature pairs with correlation greater than a threshold
threshold = 0.8
high_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if corr_matrix.iloc[i, j] > threshold:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))

# Display highly correlated pairs
if high_corr_pairs:
    print("Highly correlated feature pairs:")
    for feat1, feat2, corr in high_corr_pairs:
        print(f"{feat1} and {feat2}: {corr:.4f}")
else:
    print(f"No feature pairs with correlation above {threshold} found.")

# To remove one feature from each highly correlated pair
# (typically the one with lower correlation with target)
features_to_drop = []
for feat1, feat2, _ in high_corr_pairs:
    # Compare correlation with target
    if abs(X_train_transformed[feat1].corr(y)) < abs(X_train_transformed[feat2].corr(y)):
        features_to_drop.append(feat1)
    else:
        features_to_drop.append(feat2)

# Remove duplicates
features_to_drop = list(set(features_to_drop))
print(f"Features to drop due to high correlation: {features_to_drop}")

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class DropFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, features_to_drop):
        self.features_to_drop = features_to_drop
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        return X.drop(columns=self.features_to_drop, errors='ignore')


In [ ]:
features_to_drop = ['SaleCondition_mean', 'LandSlope_Mod', 'PavedDrive_N', 'Exterior1st_mean', 'TotRmsAbvGrd', 'Street_Pave', 'CentralAir_N', '1stFlrSF', 'GarageYrBlt', 'GarageArea', 'Utilities_NoSeWa']
a = [col for col in features_to_drop if col in X_train_transformed.columns]
pipeline_with_drop_features = Pipeline([
    ('categorical_transformer', CustomPreprocessor(mean_columns, one_hot_columns)),
    ('drop_features', DropFeatures(features_to_drop)),
    ('imputer', SimpleImputer(strategy='median')),  
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
len(a), len(features_to_drop)

In [ ]:
grid_search_with_correlated_filter = GridSearchCV(
    pipeline_with_drop_features,
    param_grid=param_grid,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='neg_mean_squared_error',
    return_train_score=True,
    n_jobs=-1,
    verbose=2
)

start_mlflow_run("KFold_simple_model", grid_search_with_correlated_filter, X_train, y_train, "KFold_with_correlation_filter")

 # preprocessor with dropping many na columns# 

In [ ]:
napreprocessor = NAPreprocessor(0.8)
X_trans = napreprocessor.fit_transform(X_train, y_train)
X_trans.info()

In [ ]:
pipeline_with_dropna_and_features = Pipeline([
    ('na_dropper', NAPreprocessor(0.8)),
    ('categorical_transformer', CustomPreprocessor(mean_columns, one_hot_columns)),
    ('drop_features', DropFeatures(features_to_drop)),
    ('imputer', SimpleImputer(strategy='median')),  
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

In [ ]:
param_grid = {

    'imputer__strategy': ['mean', 'median', 'most_frequent'],
}

grid_search_with_correlated_filter_and_nadropping = GridSearchCV(
    pipeline_with_dropna_and_features,
    param_grid=param_grid,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='neg_mean_squared_error',
    return_train_score=True,
    n_jobs=-1,
    verbose=2
)

start_mlflow_run("KFold_simple_model", grid_search_with_correlated_filter_and_nadropping, X_train, y_train, "heavy_na_dropping_5Folds" )

# RFE

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.feature_selection import RFECV
param_grid = {
    "rfe__n_features_to_select": [5, 10, 15, 20, 30]
}
rfe = RFE(estimator=LinearRegression())
rfe_pipeline = Pipeline([
    ('na_dropper', NAPreprocessor(0.8)),
    ('categorical_transformer', CustomPreprocessor(mean_columns, one_hot_columns)),
    ('drop_features', DropFeatures(features_to_drop)),
    ('imputer', SimpleImputer(strategy='median')),  
    ('scaler', StandardScaler()), 
    ('rfe', rfe),
    ('model', LinearRegression()),  
  
])
grid = GridSearchCV(
    rfe_pipeline,
    param_grid=param_grid,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="neg_mean_squared_error",
    return_train_score=True,
    n_jobs=-1
)
start_mlflow_run("KFold_simple_model", grid, X_train, y_train, "RFE_RUN_5Folds")

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.feature_selection import RFECV
param_grid = {
    "rfe__n_features_to_select": [5, 10, 15, 20, 30]
}
rfe = RFE(estimator=Ridge(random_state=42))
rfe_pipeline = Pipeline([
    ('na_dropper', NAPreprocessor(0.8)),
    ('categorical_transformer', CustomPreprocessor(mean_columns, one_hot_columns)),
    ('drop_features', DropFeatures(features_to_drop)),
    ('imputer', SimpleImputer(strategy='median')),  
    ('scaler', StandardScaler()), 
    ('rfe', rfe),
    ('model', Ridge(random_state=42)),  
  
])
grid = GridSearchCV(
    rfe_pipeline,
    param_grid=param_grid,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="neg_mean_squared_error",
    return_train_score=True,
    n_jobs=-1
)
start_mlflow_run("KFold_simple_model", grid, X_train, y_train, "RFE_RUN_5Folds_Ridge")

# lasso

In [ ]:
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel

# Parameter grid for Lasso alpha
param_grid = {
    'feature_selection__estimator__alpha': [0.001, 0.01, 0.1, 0.5, 1, 5, 10, 20, 30, 100]
}


lasso = Lasso(random_state=42, max_iter=10000)

# Pipeline with Lasso feature selection
lasso_pipeline = Pipeline([
    ('heavy_na_dropper', NAPreprocessor(0.8)),
    ('categorical_transformer', CustomPreprocessor(mean_columns, one_hot_columns)),
    ('drop_features', DropFeatures(features_to_drop)),
    ('imputer', SimpleImputer(strategy='median')),  
    ('scaler', StandardScaler()),
    ('feature_selection', SelectFromModel(lasso)),  # Lasso selects features
    ('model', LinearRegression()),  # Final model
])

grid = GridSearchCV(
    lasso_pipeline,
    param_grid=param_grid,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    return_train_score=True,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

start_mlflow_run("KFold_simple_model", grid, X_train, y_train, "LASSO_SELECTION_RUN")

# Without KFold

In [ ]:
def start_mlflow_run(name, pipeline, X, y, X_val, y_val, run_name=None):
    mlflow.set_experiment(name)
    
    with mlflow.start_run(run_name=run_name):  
        y_log = np.log1p(y)
        pipeline.fit(X, y_log)
        
        # log pipeline params
        mlflow.log_params(pipeline.get_params())

        # val metrics
        pred_val   = np.expm1(pipeline.predict(X_val))
        pred_train = np.expm1(pipeline.predict(X))

        mlflow.log_metric("train_rmse", np.sqrt(np.mean((pred_train - y) ** 2)))
        mlflow.log_metric("val_rmse",   np.sqrt(np.mean((pred_val - y_val) ** 2)))
        mlflow.log_metric("val_mae",    np.mean(np.abs(pred_val - y_val)))

        # plots
        fig1 = plot_actual_vs_predicted(y_val, pred_val)
        fig2 = plot_residuals(y_val, pred_val)
        fig3 = plot_residuals_distribution(y_val, pred_val)
        mlflow.log_figure(fig1, "plots/actual_vs_predicted.png")
        mlflow.log_figure(fig2, "plots/residuals.png")
        mlflow.log_figure(fig3, "plots/residuals_distribution.png")
        plt.close('all')

        mlflow.sklearn.log_model(pipeline, "model")

In [ ]:
from sklearn.model_selection import train_test_split

target_col = 'SalePrice'
X = df.drop(columns=[target_col])
y = df[target_col]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
s = X_train[cat_cols].nunique()
s

In [ ]:
threshold = 3
mean_columns = s[s > threshold].index
one_hot_columns = s[s<=threshold].index
mean_columns, one_hot_columns

In [ ]:
param_grid = {

    'imputer__strategy': ['mean', 'median', 'most_frequent'],
}


In [ ]:
pipeline = Pipeline([
    ('heavy-na-remover', NAPreprocessor(0.8)),
    ('categorical_transformer', CustomPreprocessor(mean_columns, one_hot_columns)),
    ('imputer', SimpleImputer(strategy='median')),  
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

In [ ]:

import mlflow
import mlflow.sklearn
import numpy as np
start_mlflow_run("Simple_Linear_Regression", pipeline,X_train, y_train, X_val, y_val, "cross_val_without_feature_selection")

# Preprocessing with smoothing

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class CustomPreprocessorWithSmoothing(BaseEstimator, TransformerMixin):
    def __init__(self, mean_columns, one_hot_columns, alpha=20):
        self.mean_columns = mean_columns # Columns Which Should Be Preprocessed Using WOE
        self.one_hot_columns = one_hot_columns # Columns Which Should Be Preprocessed Using One Hot Encoder
        self.one_hot_expected = {}
        self.alpha=alpha
    def fit(self, X, y):
        #
        print(X.shape, y.shape)
        self.mean_columns = [col for col in self.mean_columns if col in X.columns]
        self.one_hot_columns = [col for col in self.one_hot_columns if col in X.columns]
        # Generate Fill Na Values Just in Case
        self.mean_columns_fill_na = X[self.mean_columns].mode().T[0].to_dict()
        df_mean = X.copy()
        target_col = 'SalePrice'
        df_mean[target_col] = y
        self.global_mean = y.mean()
        self.mean_mappings = {}
        self.one_hot_mappings = {}
        alpha=self.alpha
        for col in self.mean_columns:
         
            groups = df_mean.groupby([col])[target_col].agg(['mean', 'count'])
            mean_dict = {}
            for category, row in groups.iterrows():
                n = row['count']
                mean = row['mean']
                smooth = (n * mean + alpha * self.global_mean) / (n + alpha)
                mean_dict[category] = smooth
            self.mean_mappings[col] = mean_dict
        for col in self.one_hot_columns : 
            unique_vals = X[col].dropna().unique()
            self.one_hot_expected[col] = [f"{col}_{val}" for val in unique_vals]
        return self


    def transform(self, X):
        X_transformed = X.copy()

        
        for col in self.mean_columns:
            X_transformed[f'{col}_mean'] = X_transformed[col].map(self.mean_mappings[col]).fillna(self.global_mean)
            X_transformed.drop(columns=col, inplace=True)

      
        for col in self.one_hot_columns:
            dummies = pd.get_dummies(
                X_transformed[col], 
                prefix=col, 
                dummy_na=True, 
                dtype=int
            )
            for expected_col in self.one_hot_expected[col]:
                if expected_col not in dummies.columns:
                    dummies[expected_col] = 0
            dummies = dummies[self.one_hot_expected[col]]
            X_transformed = pd.concat([
                X_transformed.drop(col, axis=1), 
                dummies
            ], axis=1)

        n = X_transformed.isna().mean()

        na_cols = list(n[n > 0].index)

  

        for col in na_cols:

            if not '_' in col : continue
            name, pr = col.split("_", 1)
            if pr != "mean":
                print("Error Related to Nans")

            dic = self.mean_columns_fill_na
            mappings = self.mean_mappings
           
            X_transformed[col] = X_transformed[col].fillna(mappings[name][dic[name]])
            #else X_transformed[col]
          
            
        return X_transformed

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.feature_selection import RFECV
param_grid = {
    "rfe__n_features_to_select": [5, 10, 15, 20, 30]
}
rfe = RFE(estimator=LinearRegression())
rfe_pipeline = Pipeline([
    ('na_dropper', NAPreprocessor(0.8)),
    ('categorical_transformer_with_smoothing', CustomPreprocessorWithSmoothing(mean_columns, one_hot_columns, alpha=20)),
    ('drop_features', DropFeatures(features_to_drop)),
    ('imputer', SimpleImputer(strategy='median')),  
    ('scaler', StandardScaler()), 
    ('rfe', rfe),
    ('model', LinearRegression()),  
  
])
grid = GridSearchCV(
    rfe_pipeline,
    param_grid=param_grid,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="neg_mean_squared_error",
    return_train_score=True,
    n_jobs=-1
)
start_mlflow_run_without_logtransform("KFold_simple_model_smoothing", grid, X_train, y_train, "RFE_RUN_5Folds_Smoothed_Preprocessing_alpha20_without_logtransform")

In [ ]:
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel

# Parameter grid for Lasso alpha
param_grid = {
    'feature_selection__estimator__alpha': [0.001, 0.01, 0.1, 0.5, 1, 5, 10, 20, 30, 100]
}


lasso = Lasso(random_state=42, max_iter=10000)

# Pipeline with Lasso feature selection
lasso_pipeline = Pipeline([
    ('heavy_na_dropper', NAPreprocessor(0.8)),
    ('categorical_transformer', CustomPreprocessorWithSmoothing(mean_columns, one_hot_columns)),
    ('drop_features', DropFeatures(features_to_drop)),
    ('imputer', SimpleImputer(strategy='median')),  
    ('scaler', StandardScaler()),
    ('feature_selection', SelectFromModel(lasso)),  # Lasso selects features
    ('model', LinearRegression()),  # Final model
])

grid = GridSearchCV(
    lasso_pipeline,
    param_grid=param_grid,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    return_train_score=True,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

start_mlflow_run("KFold_simple_model_smoothing", grid, X_train, y_train, "LASSO_SELECTION_RUN_Smoothing")

# RFE without KFold

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.feature_selection import RFECV
param_grid = {
    "rfe__n_features_to_select": [5, 10, 15, 20, 30]
}
rfe = RFE(estimator=LinearRegression(), n_features_to_select=5)
rfe_pipeline = Pipeline([
    ('na_dropper', NAPreprocessor(0.8)),
    ('categorical_transformer', CustomPreprocessor(mean_columns, one_hot_columns)),
   
    ('imputer', SimpleImputer(strategy='median')),  
    ('scaler', StandardScaler()), 
    ('rfe', rfe),
    ('model', LinearRegression()),  
  
])

start_mlflow_run("Simple_Linear_Regression", rfe_pipeline, X_train, y_train, X_val,y_val, "RFE_RUN_5_features")

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.feature_selection import RFECV
param_grid = {
    "rfe__n_features_to_select": [5, 10, 15, 20, 30]
}
rfe = RFE(estimator=LinearRegression(), n_features_to_select=5)
rfe_pipeline = Pipeline([
    ('na_dropper', NAPreprocessor(0.8)),
    ('categorical_transformer', CustomPreprocessorWithSmoothing(mean_columns, one_hot_columns, alpha=20)),
   
    ('imputer', SimpleImputer(strategy='median')),  
    ('scaler', StandardScaler()), 
    ('rfe', rfe),
    ('model', LinearRegression()),  
  
])

start_mlflow_run("Simple_Linear_Regression", rfe_pipeline, X_train, y_train, X_val,y_val, "RFE_RUN_5_features")

# Outlier Detection for KFold 

In [ ]:
from sklearn.base import clone
def start_mlflow_run_outliers(name, grid_search_arg, X, y, run_name=None):
    mlflow.set_experiment(name)
    
    with mlflow.start_run(run_name=run_name):  
        y_log = np.log1p(y)
        grid_search_arg.fit(X, y_log)
        best_model = grid_search_arg.best_estimator_
        mlflow.log_params(grid_search_arg.best_params_)
        kf = grid_search_arg.cv

        preds = np.zeros_like(y, dtype=float)
        i = 0
        errors = np.zeros(len(y))
        for train_idx, val_idx in kf.split(X):
            model = clone(best_model)
            
            model.fit(X.iloc[train_idx], y_log.iloc[train_idx])
            
            pred_log = model.predict(X.iloc[val_idx])
            preds[val_idx] = np.expm1(pred_log)
            errors[val_idx]=np.abs(preds[val_idx] - y.iloc[val_idx])
            i += 1
        hard_samples = errors.argsort()[-20:]
        df_hard = pd.DataFrame({
            "index": hard_samples,
            "error": errors[hard_samples],
            "y_true": y.iloc[hard_samples].values,
            "y_pred": preds[hard_samples]
        })
        
        df_hard.to_csv("hard_samples.csv", index=False)
        mlflow.log_artifact("hard_samples.csv")
        # metrics
        n_splits = grid_search_arg.cv.get_n_splits()
        train_scores = [
            -grid_search_arg.cv_results_[f"split{i}_train_score"][grid_search_arg.best_index_]
            for i in range(n_splits)
        ]
        
        val_scores = [
            -grid_search_arg.cv_results_[f"split{i}_test_score"][grid_search_arg.best_index_]
            for i in range(n_splits)
        ]
        mlflow.log_metric("cv_mse_train_mean", np.mean(train_scores))
        mlflow.log_metric("cv_mse_val_mean", np.mean(val_scores))
        mlflow.log_metric("cv_mse_std", np.std(val_scores))


        # plots in real scale
        y_pred_log = cross_val_predict(
            grid_search_arg.best_estimator_,
            X,
            y_log,
            cv=grid_search_arg.cv,
            n_jobs=-1
        )
        y_pred = np.expm1(y_pred_log)
        y_real = y
        fig1 = plot_actual_vs_predicted(y_real, y_pred)
        fig2 = plot_residuals(y_real, y_pred)
        fig3 = plot_residuals_distribution(y_real, y_pred)
        mlflow.log_figure(fig1, "plots/actual_vs_predicted.png")
        mlflow.log_figure(fig2, "plots/residuals.png")
        mlflow.log_figure(fig3, "plots/residuals_distribution.png")
        plt.close('all')

        mlflow.sklearn.log_model(best_model, "model")

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.feature_selection import RFECV
param_grid = {
    "rfe__n_features_to_select": [5, 10, 15, 20, 30]
}
rfe = RFE(estimator=LinearRegression())
rfe_pipeline = Pipeline([
    ('na_dropper', NAPreprocessor(0.8)),
    ('categorical_transformer', CustomPreprocessor(mean_columns, one_hot_columns)),
    ('drop_features', DropFeatures(features_to_drop)),
    ('imputer', SimpleImputer(strategy='median')),  
    ('scaler', StandardScaler()), 
    ('rfe', rfe),
    ('model', LinearRegression()),  
  
])
grid = GridSearchCV(
    rfe_pipeline,
    param_grid=param_grid,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="neg_mean_squared_error",
    return_train_score=True,
    n_jobs=-1
)
start_mlflow_run_outliers("Detecting outliers", grid, X_train, y_train, "RFE_RUN_5Folds")

In [ ]:
df_hard_samples = pd.read_csv("/kaggle/input/datasets/tarashdolaberidze/hard-samples/hard_samples.csv")
df_hard_samples.info()

In [ ]:
errors = df_hard_samples["error"]
errors.describe()

In [ ]:
df_hard_samples[outliers]["index"]

In [ ]:
bad = X.loc[df_hard_samples["index"]].index
bad

In [ ]:
pipeline = Pipeline([
    ('na_dropper', NAPreprocessor(0.8)),
    ('categorical_transformer', CustomPreprocessor(mean_columns, one_hot_columns)),
   
    ('imputer', SimpleImputer(strategy='median')),  
    ('scaler', StandardScaler()), 
  
])

In [ ]:
X_transformed = pipeline.fit_transform(X, y)
X_transformed.shape

In [ ]:
from sklearn.metrics import pairwise_distances

worst_X = X_transformed[bad]

dist = pairwise_distances(worst_X)
avg_worst = dist.mean()
avg_worst

In [ ]:
import numpy as np
from sklearn.metrics import pairwise_distances

rand_idx = np.random.choice(X_transformed.shape[0], len(bad), replace=False)
random_X = X_transformed[rand_idx]

avg_random = pairwise_distances(random_X).mean()
avg_random

In [ ]:
mlflow.set_experiment("Detecting outliers") 
with mlflow.start_run(run_name="Anomaly Scarcity") : 
    from sklearn.metrics import pairwise_distances

    worst_X = X_transformed[bad]
    
    dist = pairwise_distances(worst_X)
    avg_worst = dist.mean()
    avg_worst
    mlflow.log_metric("Big Error Average Distance", avg_worst)
    for i in range(0, 20) : 
        rand_idx = np.random.choice(X_transformed.shape[0], len(bad), replace=False)
        random_X = X_transformed[rand_idx]
        
        avg_random = pairwise_distances(random_X).mean()
        mlflow.log_metric(f"Random Distance {i}", avg_random)
        

In [ ]:
df_hard_samples["y_true"] - df_hard_samples["y_pred"]

# Regression Trees 

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.tree import DecisionTreeRegressor

param_grid = {
    
    "model__min_samples_leaf": [1, 5, 10, 20],
    "model__min_samples_split": [2, 10, 20]
}

decision_tree_pipeline = Pipeline([
    ('na_dropper', NAPreprocessor(0.8)),
    ('categorical_transformer', CustomPreprocessor(mean_columns, one_hot_columns)),
   
    ('imputer', SimpleImputer(strategy='median')),  
    ('model', DecisionTreeRegressor(max_depth=10, random_state=42)),  
  
])
grid = GridSearchCV(
    decision_tree_pipeline,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    return_train_score=True,
    n_jobs=-1
)
start_mlflow_run("Decision Tree", grid, X_train, y_train, "Regression_tree_depth10")


In [ ]:
from sklearn.base import clone
def start_mlflow_run_without_logtransform(name, grid_search_arg, X, y, run_name=None):
    mlflow.set_experiment(name)
    
    with mlflow.start_run(run_name=run_name):  
     
        grid_search_arg.fit(X, y)
        best_model = grid_search_arg.best_estimator_
        mlflow.log_params(grid_search_arg.best_params_)
        kf = grid_search_arg.cv

        preds = np.zeros_like(y, dtype=float)
        i = 0
        for train_idx, val_idx in kf.split(X):
            model = clone(best_model)
            
            model.fit(X.iloc[train_idx], y.iloc[train_idx])
            
            preds[val_idx] = model.predict(X.iloc[val_idx])
         
            mlflow.log_metric(f"split {i} RMSE", np.sqrt(np.mean((preds[val_idx] - y.iloc[val_idx])**2)))
            i += 1
        # metrics
        n_splits = grid_search_arg.cv.get_n_splits()
        train_scores = [
            -grid_search_arg.cv_results_[f"split{i}_train_score"][grid_search_arg.best_index_]
            for i in range(n_splits)
        ]
        
        val_scores = [
            -grid_search_arg.cv_results_[f"split{i}_test_score"][grid_search_arg.best_index_]
            for i in range(n_splits)
        ]
        mlflow.log_metric("cv_mse_train_mean", np.mean(train_scores))
        mlflow.log_metric("cv_mse_val_mean", np.mean(val_scores))
        mlflow.log_metric("cv_mse_std", np.std(val_scores))


        # plots in real scale
        y_pred= cross_val_predict(
            grid_search_arg.best_estimator_,
            X,
            y,
            cv=grid_search_arg.cv,
            n_jobs=-1
        )
        y_real=y
        fig1 = plot_actual_vs_predicted(y_real, y_pred)
        fig2 = plot_residuals(y_real, y_pred)
        fig3 = plot_residuals_distribution(y_real, y_pred)
        mlflow.log_figure(fig1, "plots/actual_vs_predicted.png")
        mlflow.log_figure(fig2, "plots/residuals.png")
        mlflow.log_figure(fig3, "plots/residuals_distribution.png")
        plt.close('all')

        mlflow.sklearn.log_model(best_model, "model")

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.tree import DecisionTreeRegressor

param_grid = {
    
    "model__min_samples_leaf": [1, 5, 10, 20],
    "model__min_samples_split": [2, 10, 20]
}

decision_tree_pipeline = Pipeline([
    ('na_dropper', NAPreprocessor(0.8)),
    ('categorical_transformer', CustomPreprocessor(mean_columns, one_hot_columns)),
   
    ('imputer', SimpleImputer(strategy='median')),  
    ('model', DecisionTreeRegressor(max_depth=5, random_state=42)),  
  
])
grid = GridSearchCV(
    decision_tree_pipeline,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    return_train_score=True,
    n_jobs=-1
)
start_mlflow_run_without_logtransform("Decision Tree", grid, X_train, y_train, "Regression_tree_nologtransform")


In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.tree import DecisionTreeRegressor

param_grid = {
    
    "model__min_samples_leaf": [1, 5, 10, 20],
    "model__min_samples_split": [2, 10, 20]
}

decision_tree_pipeline = Pipeline([
    ('na_dropper', NAPreprocessor(0.8)),
    ('categorical_transformer', CustomPreprocessorWithSmoothing(mean_columns, one_hot_columns)),
   
    ('imputer', SimpleImputer(strategy='median')),  
    ('model', DecisionTreeRegressor(max_depth=5, random_state=42)),  
  
])
grid = GridSearchCV(
    decision_tree_pipeline,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    return_train_score=True,
    n_jobs=-1
)
start_mlflow_run_without_logtransform("Decision Tree Smoothing", grid, X_train, y_train, "Regression_tree_nologtransform")


In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.tree import DecisionTreeRegressor

param_grid = {
    
    "model__min_samples_leaf": [1, 5, 10, 20],
    "model__min_samples_split": [2, 10, 20]
}

decision_tree_pipeline = Pipeline([
    ('na_dropper', NAPreprocessor(0.8)),
    ('categorical_transformer', CustomPreprocessorWithSmoothing(mean_columns, one_hot_columns)),
   
    ('imputer', SimpleImputer(strategy='median')),  
    ('model', DecisionTreeRegressor(max_depth=10, random_state=42)),  
  
])
grid = GridSearchCV(
    decision_tree_pipeline,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    return_train_score=True,
    n_jobs=-1
)
start_mlflow_run("Decision Tree Smoothing", grid, X_train, y_train, "Regression_tree_depth10")


# Random Forest 

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor

param_grid = {
    
    
    "model__min_samples_split": [2, 10, 20]
}

decision_tree_pipeline = Pipeline([
    ('na_dropper', NAPreprocessor(0.8)),
    ('categorical_transformer', CustomPreprocessorWithSmoothing(mean_columns, one_hot_columns)),
    
    ('imputer', SimpleImputer(strategy='median')),  
    ("model", RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=10,
        n_jobs=-1,
        random_state=42
    ))
  
])
grid = GridSearchCV(
    decision_tree_pipeline,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    return_train_score=True,
    n_jobs=-1
)
start_mlflow_run_without_logtransform("Random Forest", grid, X_train, y_train, "RandomForest_nologtransform_n_est500")


In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor

param_grid = {
    
    
    "model__min_samples_split": [2, 10, 20]
}

decision_tree_pipeline = Pipeline([
    ('na_dropper', NAPreprocessor(0.8)),
    ('categorical_transformer', CustomPreprocessorWithSmoothing(mean_columns, one_hot_columns)),
    
    ('imputer', SimpleImputer(strategy='median')),  
    ("model", RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=10,
        n_jobs=-1,
        random_state=42
    ))
  
])
grid = GridSearchCV(
    decision_tree_pipeline,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    return_train_score=True,
    n_jobs=-1
)
start_mlflow_run("Random Forest", grid, X_train, y_train, "RandomForest_logtransform_n_est500")


# Feature Engineering 
It's clear that the real bottle neck is that we lack nonlinearity in features. Here we try to build Pipeline which will give us log, sqrt and square for each initial numerical columns

In [ ]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        self.num_cols = X.select_dtypes(include=np.number).columns
        return self

    def transform(self, X):
        X_transformed = X.copy()

        for col in self.num_cols:
            x = X_transformed[col]

            X_transformed[f"{col}_log"] = np.log1p(x.clip(lower=0))

            X_transformed[f"{col}_sqrt"] = np.sqrt(x.clip(lower=0))

            X_transformed[f"{col}_square"] = x ** 2

        return X_transformed

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.feature_selection import RFECV
param_grid = {
    "rfe__n_features_to_select": [5, 10, 15, 20, 30]
}
rfe = RFE(estimator=LinearRegression())
rfe_pipeline = Pipeline([
    ('na_dropper', NAPreprocessor(0.8)),
    ('categorical_transformer', CustomPreprocessor(mean_columns, one_hot_columns)),
    ('drop_features', DropFeatures(features_to_drop)),
    ('feature_eng', FeatureEngineer()),
    ('imputer', SimpleImputer(strategy='median')),  
    ('scaler', StandardScaler()), 
    ('rfe', rfe),
    ('model', LinearRegression()),  
  
])
grid = GridSearchCV(
    rfe_pipeline,
    param_grid=param_grid,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="neg_mean_squared_error",
    return_train_score=True,
    n_jobs=-1
)
start_mlflow_run("Feature_engineering", grid, X_train, y_train, "RFE_RUN_5Folds_feature_engineering")